# 01 — Python Analysis (EDA, Trends, CPR)
**Zakres:** wczytanie danych, sanity-checki, agregacje Pandas, wykresy, eksporty.
_Uwaga:_ SQL/SQLite dorzucimy w etapie 2.

In [ ]:

# Importy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Ścieżki relatywne
BASE = Path("..").resolve().parent if (Path.cwd().name == "notebooks") else Path(".")
DATA = BASE / "data" / "data.csv"
FIG = BASE / "figures"
OUT = BASE / "outputs"
FIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Wczytanie
df = pd.read_csv(DATA)
df['day_id'] = pd.to_datetime(df['day_id'])
df.head()


In [ ]:

# Informacje o danych
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nZakres dat:", df['day_id'].min(), "→", df['day_id'].max())

for col in ['license','model','feature']:
    print(f"\nUnikalne w {col}:", df[col].nunique())
    print(df[col].value_counts().head(10))


## Agregacje bazowe

In [ ]:

by_model = df.groupby('model', as_index=False).agg(
    requests_sum=('requests_cnt','sum'),
    spent_sum=('spent_amount','sum'),
    users=('uuid','nunique')
).sort_values('requests_sum', ascending=False)

by_feature = df.groupby('feature', as_index=False).agg(
    requests_sum=('requests_cnt','sum'),
    spent_sum=('spent_amount','sum'),
    users=('uuid','nunique')
).sort_values('requests_sum', ascending=False)

by_license = df.groupby('license', as_index=False).agg(
    requests_sum=('requests_cnt','sum'),
    spent_sum=('spent_amount','sum'),
    users=('uuid','nunique')
).sort_values('requests_sum', ascending=False)

by_model.to_csv(OUT/'by_model.csv', index=False)
by_feature.to_csv(OUT/'by_feature.csv', index=False)
by_license.to_csv(OUT/'by_license.csv', index=False)

by_model, by_feature, by_license


## Związek requests ↔ spent (korelacja + scatter)

In [ ]:

corr = df[['requests_cnt','spent_amount']].corr().loc['requests_cnt','spent_amount']
print(f"Korelacja requests_cnt ↔ spent_amount: {corr:.3f}")

plt.figure()
plt.scatter(df['requests_cnt'], df['spent_amount'], s=2, alpha=0.2)
plt.title('Requests vs Spent')
plt.xlabel('requests_cnt')
plt.ylabel('spent_amount')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG/'scatter_requests_spent.png', dpi=150)
plt.show()


## Trendy tygodniowe

In [ ]:

week = df['day_id'].dt.to_period('W').astype(str)
trend_week = df.groupby(week, as_index=False).agg(
    requests_sum=('requests_cnt','sum'),
    spent_sum=('spent_amount','sum'),
    users=('uuid','nunique')
).rename(columns={'day_id':'week'}).sort_values('week')

trend_week.to_csv(OUT/'trend_week.csv', index=False)

plt.figure()
plt.plot(trend_week['week'], trend_week['requests_sum'])
plt.title('Suma requests — tygodniowo')
plt.xlabel('tydzień')
plt.ylabel('suma requests')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG/'trend_week_requests.png', dpi=150)
plt.show()

plt.figure()
plt.plot(trend_week['week'], trend_week['spent_sum'])
plt.title('Suma spent — tygodniowo')
plt.xlabel('tydzień')
plt.ylabel('suma spent')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG/'trend_week_spent.png', dpi=150)
plt.show()

trend_week.head()


## Kombinacje (model, feature) i (feature, license)

In [ ]:

mf = df.groupby(['model','feature'], as_index=False).agg(
    requests_sum=('requests_cnt','sum'),
    spent_sum=('spent_amount','sum')
).sort_values('requests_sum', ascending=False)

fl = df.groupby(['feature','license'], as_index=False).agg(
    requests_sum=('requests_cnt','sum'),
    spent_sum=('spent_amount','sum')
).sort_values('requests_sum', ascending=False)

mf.to_csv(OUT/'by_model_feature.csv', index=False)
fl.to_csv(OUT/'by_feature_license.csv', index=False)

mf.head(10), fl.head(10)


## Pivot udziałów feature w ramach modelu + „heatmapa” (matplotlib)

In [ ]:

pivot_model_feature = df.pivot_table(index='model', columns='feature', values='requests_cnt', aggfunc='sum').fillna(0)
pivot_share = pivot_model_feature.div(pivot_model_feature.sum(axis=1), axis=0)

import numpy as np
plt.figure()
plt.imshow(pivot_share.values, aspect='auto')
plt.title('Udział requests per feature w ramach modelu')
plt.yticks(ticks=range(len(pivot_share.index)), labels=pivot_share.index)
plt.xticks(ticks=range(len(pivot_share.columns)), labels=pivot_share.columns, rotation=45, ha='right')
plt.colorbar()
plt.tight_layout()
plt.savefig(FIG/'heatmap_model_feature_share.png', dpi=150)
plt.show()

pivot_share.round(3)


## Efektywność — spent per request (CPR)

In [ ]:

def cpr(group_cols):
    res = (df.groupby(group_cols, as_index=False)
             .agg(requests_sum=('requests_cnt','sum'),
                  spent_sum=('spent_amount','sum'),
                  users=('uuid','nunique')))
    res['spent_per_request'] = res['spent_sum'] / res['requests_sum']
    return res.sort_values('spent_per_request', ascending=False)

eff_model = cpr(['model'])
eff_feature = cpr(['feature'])
eff_license = cpr(['license'])

eff_model.to_csv(OUT/'eff_model.csv', index=False)
eff_feature.to_csv(OUT/'eff_feature.csv', index=False)
eff_license.to_csv(OUT/'eff_license.csv', index=False)

eff_model.head(10)
